In [ ]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

## Prerequisites

In [ ]:
import pandas as pd
import pickle

from radp.digital_twin.utils.gis_tools import GISTools
from notebooks.radp_library import calculate_received_power, get_ues_cells_cartesian_df, calc_rx_power, calc_log_distance, calc_relative_bearing, preprocess_ue_data
from notebooks.radp_library import get_percell_data

# Curating training/update data

training/update data: key value pairs of cell_id vs processed df (have engineered features). format:

```bash
{
    "cell_1": df1,
    "cell_2": df2,
    "cell_3": df3,
    ...      
}
```

In [ ]:
# place the pkl file in the same directory as this script, downloadable from maveric drive

training_pickle_path = Path('notebooks/data/sim_data/processed_training_data.pkl')
absolute_pickle_path =Path().absolute().parent / training_pickle_path
with open(absolute_pickle_path, 'rb') as f:
    training_data = pickle.load(f)

print(training_data)

# loading ue and topology

In [ ]:
# place the required files in the same directory as this script, downloadable from maveric drive

simple_ue = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')

In [ ]:
topology

In [ ]:
simple_ue = simple_ue.loc[:, ['longitude', 'latitude']]
simple_ue.head()

# Functions

## Preprocess functions

check inside `radp_library.py`:
1. get_ues_cells_cartesian_df
2. calc_log_distance
3. calcalculate_received_power
4. calc_relative_bearing
5. preprocess_ue_data (calls above 1-3)

**used _ in function names to separate from radp library import**

In [ ]:
# f0

def _get_ues_cells_cartesian_df(data,topology):
    if topology["cell_id"].dtype == object:
            topology["cell_id"] = (
                topology["cell_id"].str.replace("cell_", "").astype(int)
            )
    data["key"] = 1
    topology["key"] = 1
    cartesian_df = pd.merge(data, topology, on="key").drop("key", axis=1)
    
    data.drop(columns=["key"], inplace=True)
    topology.drop(columns=["key"], inplace=True)
    return cartesian_df

In [ ]:
# f1

def _calc_log_distance(cartesian_df):
    cartesian_df["log_distance"] = cartesian_df.apply(
        lambda row: GISTools.get_log_distance(
            row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
        ),
        axis=1,
    )
    return cartesian_df

In [ ]:
# f2

def _calc_rx_power(cartesian_df):
    cartesian_df["cell_rxpwr_dbm"] = cartesian_df.apply(
        lambda row: calculate_received_power(
            row["log_distance"], row["cell_carrier_freq_mhz"]
        ),
        axis=1,
    )
    return cartesian_df

In [ ]:
# f3

def _calc_relative_bearing(cartesian_df):
    cartesian_df["relative_bearing"] = cartesian_df.apply(
        lambda row: GISTools.get_relative_bearing(
            row["cell_az_deg"],
            row["cell_lat"],
            row["cell_lon"],
            row["latitude"],
            row["longitude"],
        ),
        axis=1,
    )
    return cartesian_df

In [ ]:
# f4

def _preprocess_ue_data(data, topology):
    cartesian_df = get_ues_cells_cartesian_df(data, topology)
    cartesian_df = calc_log_distance(cartesian_df)
    return calc_rx_power(cartesian_df)


## Prepare train or update data function

prepares key value pair data format for training or updating

In [ ]:
def prepare_train_or_update_data(df):
    update_data = calc_log_distance(df)
    update_data = calc_relative_bearing(update_data)
    update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

    train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
    n_cell = len(topology.index)

    metadata_df = pd.DataFrame(
        {
            "cell_id": [cell_id for cell_id in topology.cell_id],
            "idx": [i + 1 for i in range(n_cell)],
        }
    )
    
    idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
    n_samples_train = []
    
    for df in train_per_cell_df:
        n_samples_train.append(df.shape[0])

    train_per_cell_df_processed = []
    for i in range(n_cell):
        train_per_cell_df_processed.append(
            get_percell_data(
                data_in=train_per_cell_df[i],
                choose_strongest_samples_percell=False,
                n_samples=n_samples_train[i],
            )[0][0]
        )

    training_data = {}

    for i, df in enumerate(train_per_cell_df_processed):
        train_cell_id = idx_cell_id_mapping[i + 1]
        training_data[train_cell_id] = df
    
    return training_data

## Bebugging

classic pranto moment :p

In [ ]:
simple_ue.head()

In [ ]:
topology

In [ ]:
cartesian_df = _get_ues_cells_cartesian_df(simple_ue, topology)
cartesian_df

In [ ]:
cartesian_df = _calc_log_distance(cartesian_df)
cartesian_df

In [ ]:
cartesian_df = _calc_rx_power(cartesian_df)
cartesian_df

In [ ]:
# this call should be same as previous df but with different function call

cartesian_df = _preprocess_ue_data(simple_ue, topology)
cartesian_df

In [ ]:
cartesian_df = _calc_relative_bearing(cartesian_df)
cartesian_df

## Rewriting Update (Training From Scratch)

In [ ]:
simple_ue = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')

In [ ]:
simple_ue = simple_ue.loc[:, ['longitude', 'latitude']]

simple_ue.head()

In [ ]:
topology

In [ ]:
# assume user/client/developer has the UE data withour rx power information. should call the following function to get the rx power information

df = preprocess_ue_data(simple_ue, topology) # returns the rx power information in cartesian format
df1 = df.copy()

In [ ]:
df.head()

In [ ]:
bayesian_digital_twins = {}

In [ ]:
try:
    if not isinstance(df, pd.DataFrame):
        raise TypeError("The input 'new_data' must be a pandas DataFrame.")

    expected_columns = {"longitude", "latitude", "cell_lat", "cell_lon", "cell_id", "cell_az_deg", "cell_carrier_freq_mhz", "cell_rxpwr_dbm"}
    if not expected_columns.issubset(df.columns):
        raise ValueError(
            f"The input DataFrame must contain the following columns: {expected_columns}"
        )
    
    # ? do we need str cell_id or int cell_id? does both work?
    df["cell_id"] = df["cell_id"].apply(lambda x: f"cell_{x}")
    topology["cell_id"] = topology["cell_id"].apply(lambda x: f"cell_{x}")
    print(df)
    
    prepared_data = prepare_train_or_update_data(df)
    print(prepared_data)
    if bayesian_digital_twins:
        # TODO: Update BDT
        pass
    else:
        # TODO: Create BDT from scratch
        pass

        # TODO: Add Train

except TypeError as te:
    print(f"TypeError: {te}")
except ValueError as ve:
    print(f"ValueError: {ve}")
except KeyError as ke:
    print(f"KeyError: {ke}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### Testing

In [ ]:
update_data = calc_log_distance(df1)
update_data = calc_relative_bearing(update_data)

update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

In [ ]:
update_data

In [ ]:
train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
n_cell = len(topology.index)

metadata_df = pd.DataFrame(
    {
        "cell_id": [cell_id for cell_id in topology.cell_id],
        "idx": [i + 1 for i in range(n_cell)],
    }
)
idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
desired_idxs = [1 + r for r in range(n_cell)]

n_samples_train = []
for df in train_per_cell_df:
    n_samples_train.append(df.shape[0])

train_per_cell_df_processed = []
for i in range(n_cell):
    train_per_cell_df_processed.append(
        get_percell_data(
            data_in=train_per_cell_df[i],
            choose_strongest_samples_percell=False,
            n_samples=n_samples_train[i],
        )[0][0]
    )

training_data = {}

for i, df in enumerate(train_per_cell_df_processed):
    train_cell_id = idx_cell_id_mapping[i + 1]
    training_data[train_cell_id] = df

In [ ]:
training_data

In [ ]:
print(loaded_dict)